## Topic 4: Indexing & HNSW Explained

### Concept, Intuition, Why It Exists

**What is an index, and why does it exist at all?**

- When you store vectors in a flat list and search it, you are doing **brute-force search** — computing the similarity between your query vector and every single stored vector, one by one, then sorting to find the top-k. This is called an **exact nearest neighbor search** — it is guaranteed to return the true closest vectors, no approximations.
- The problem is that brute-force search scales linearly: double the number of stored vectors, double the search time. At 1,000 vectors this is invisible. At 1,000,000 vectors this is seconds per query. At 100,000,000 vectors this is completely infeasible.
- An **index** is a data structure built over your vectors, before any queries arrive, that reorganizes them in a way that lets you skip most of the comparisons during search. Instead of checking every vector, the index lets you check only a small, intelligently selected subset — vectors that are likely to be close to the query — and still return results that are almost always correct.
- The trade-off is fundamental and explicit: **exact search** (check everything, guaranteed correct, slow at scale) vs. **approximate nearest neighbor (ANN) search** (check a smart subset, occasionally misses the single closest vector, fast at any scale). Almost every production vector search system accepts this trade-off — the occasional miss is worth the orders-of-magnitude speed gain.
- This project's earlier in-memory list of vectors used brute-force exact search. Moving to Qdrant means moving to indexed ANN search — the same conceptual operation, done in a fundamentally different and scalable way.

**What is HNSW?**

- **Hierarchical Navigable Small World** is the index structure Qdrant uses by default, and the most widely used ANN index in production vector databases (also used internally by FAISS, Weaviate, and others).
- The name describes exactly what it is: a **graph** of vectors, organized in **multiple layers** (hierarchical), where each node is connected to a small number of nearby neighbors (navigable small world), and search means traversing that graph to find the closest nodes to a query — rather than scanning a flat list.

### Internal Working

**How HNSW is built (the index construction phase):**

1. Vectors are inserted one at a time into the graph. Each vector becomes a **node**.
2. Each new node is assigned a **maximum layer** by a random process — most nodes exist only in layer 0 (the bottom, densest layer), fewer in layer 1, fewer still in layer 2, and so on. This exponentially thinning distribution across layers is what makes the structure hierarchical.
3. In each layer the node exists in, it is connected to its `m` nearest neighbors **at that layer** — `m` is a key parameter (typically 16–64) that controls the graph's density. More connections = better recall, more memory.
4. The result is a multi-layer graph where the top layers have few nodes and long-range connections (good for quickly navigating to the right general region of the space), and the bottom layer (layer 0) has all nodes and short-range connections (good for precise local search once you're in the right region).

**How HNSW search works (the query phase):**

1. Start at the entry point — a fixed node in the topmost layer.
2. **Greedy traversal downward**: at each layer, follow edges to whichever neighbor is closest to the query vector, moving greedily toward the query until no neighbor is closer than the current node. This is a local greedy search — fast, but approximate.
3. Drop down to the next layer, using the best node found in the layer above as the new starting point. Repeat.
4. At layer 0 (the densest layer with all nodes), do a more thorough local search using a parameter called `ef` (the search-time exploration factor) — explore `ef` candidate neighbors instead of just greedily following the single best at each step. Higher `ef` = better recall, slower search.
5. Return the top-k nodes from the layer 0 search as the approximate nearest neighbors.

**Why the hierarchical structure helps:**

- Top layers act like a coarse map — they let search jump quickly to the right region of the space without examining every node.
- Bottom layers act like a detailed local search — once you're in the right region, you search carefully among nearby nodes.
- This is exactly analogous to how you'd search a city: first navigate to the right neighborhood (top layers), then look carefully for the specific address (bottom layer). Checking every address in the city from the start would be brute-force; using the neighborhood structure is HNSW.

**Key parameters:**

- `m`: number of edges per node per layer. Higher = better recall, more memory, slower build. Default 16, production systems often use 32–64.
- `ef_construction`: how thoroughly to search for neighbors during index build. Higher = better index quality, slower build time. Default 100.
- `ef` (search time): how thoroughly to search at query time. Higher = better recall, slower query. Can be tuned per-query without rebuilding the index.

### How It's Implemented In This Project

- Qdrant uses HNSW by default for every collection. When `create_collection()` is called with `VectorParams(size=384, distance=Distance.COSINE)`, Qdrant automatically builds and maintains an HNSW index over all upserted vectors — no explicit index construction step needed, it happens incrementally as points are added.
- The default `m` and `ef_construction` values Qdrant uses are sensible for most corpora. At this project's scale (hundreds of chunks, not millions), the index parameters make no measurable difference — they matter when the corpus grows into the tens of thousands of vectors.
- Qdrant's payload filtering during search (used in the metadata filtering and PII topics later) is implemented by constraining which nodes the HNSW traversal is allowed to visit — only nodes whose payload matches the filter are valid candidates. This is more efficient than post-search filtering because the traversal never wastes time on non-matching nodes.

### Real-World Issues, Edge Cases, Debugging

- **ANN recall is not 100%**: HNSW will occasionally miss the single true nearest neighbor, returning the second or third closest instead. For RAG, this is almost always acceptable — the difference between the first and second closest chunk is rarely the difference between a correct and incorrect answer. For applications requiring provably exact results (fraud detection thresholds, legal document matching), ANN is not appropriate without additional exact-verification steps.
- **Index build time vs. query time trade-off**: high `ef_construction` and `m` produce a better quality index but take longer to build and use more memory. For a batch ingest of a large corpus, this matters. For this project's scale, it doesn't.
- **Filtering and recall interact**: when payload filtering is applied during HNSW search, the effective search space shrinks. If the filter is very restrictive (e.g. "only search 10 points out of 10,000"), HNSW traversal may struggle to find enough valid candidates and return fewer than k results. Qdrant handles this gracefully but it's worth knowing that aggressive filtering on small subsets can reduce effective recall.
- **HNSW is not updateable in place at low cost**: deleting a node from an HNSW graph doesn't cleanly remove it — Qdrant marks it as deleted and excludes it from results, but the graph edges remain, slightly degrading traversal efficiency over time. Periodic re-indexing (rebuilding the collection) recovers that efficiency for large-scale systems with many deletions.

### Design Decisions & Trade-offs

- HNSW vs. flat (brute-force exact search): flat search is actually the right choice at very small scale (hundreds of vectors) because there's no index build overhead and exact results are guaranteed. HNSW becomes the right choice as the corpus grows, accepting occasional misses in exchange for sub-linear search time. Qdrant uses HNSW by default regardless of collection size, which means at very small scales you're paying a small build overhead for an index you don't strictly need — a sensible default for a system that expects to grow.
- HNSW vs. IVF (Inverted File Index, used by FAISS): IVF partitions vectors into clusters (using k-means) and at search time only examines the clusters closest to the query. IVF is faster to build than HNSW and uses less memory, but has worse recall at the same search budget. HNSW has better recall for the same query time, at the cost of more memory. For most RAG workloads, recall matters more than memory, so HNSW is the better default.

### Alternatives & When To Use Each

- Flat / brute-force exact search — corpus fits in memory, exact results required, scale is small enough that linear scan is fast. Qdrant supports this explicitly with `VectorParams(on_disk=False)` and no HNSW — useful for tiny reference collections.
- HNSW (Qdrant default) — general-purpose production default, excellent recall, scales well, higher memory than IVF.
- IVF (FAISS) — very large corpora where memory is the primary constraint and slightly lower recall is acceptable.
- ScaNN, DiskANN — specialized systems for billion-scale corpora or on-disk indexing where even HNSW's memory requirements become prohibitive.

### Common Mistakes & Production Failures

- Treating ANN recall as binary ("it either works or it doesn't") rather than as a tunable parameter — increasing `ef` at query time directly improves recall at the cost of slightly slower search, and this trade-off can be adjusted without rebuilding the index.
- Expecting HNSW to handle very aggressive metadata filtering well on tiny subsets — if the filter leaves only 5 valid points in a 50,000-point collection, HNSW will struggle to find them efficiently, and a filtered brute-force scan of those 5 points would actually be faster and more correct.
- Not accounting for HNSW memory usage when sizing infrastructure — an HNSW index for 10 million 384-dimensional vectors can consume several GB of RAM beyond the raw vector storage, depending on `m`.

### Lead-Level Interview Questions

**Q: Why is approximate nearest neighbor search acceptable for RAG, but might not be acceptable for other applications?**
A: In RAG, the goal is to retrieve chunks that are relevant enough for the LLM to produce a correct answer. The difference between the first and second closest chunk is rarely meaningful — both are likely relevant. In contrast, applications like fraud detection (is this transaction within a threshold distance of known fraud patterns?) or biometric matching (is this face embedding the closest match to a stored identity?) may require provably exact nearest neighbors, because the specific closest result, not just a close one, is the decision boundary.

**Q: HNSW occasionally misses the true nearest neighbor. How would you detect whether this is actually impacting retrieval quality in production?**
A: Build a small evaluation set of queries with known correct chunks (the chunks that should be retrieved for each query, verified by hand or by exact search). Run production HNSW search and measure recall at k — the fraction of queries where the known correct chunk appears in the top-k results. If recall is below acceptable, increase `ef` at search time, which improves recall without rebuilding the index. If that's insufficient, consider increasing `m` and rebuilding.

**Q: A collection has 10,000 points but a metadata filter restricts search to 20 of them. HNSW returns fewer than the requested k results. Why, and what's the fix?**
A: HNSW traversal is guided by the graph structure built over all 10,000 points — it navigates toward the query through the full graph, but can only collect results from the 20 matching points. If the graph's traversal path doesn't naturally pass through enough of those 20 points within the `ef` search budget, it returns fewer than k. The fix is either increasing `ef` specifically for heavily-filtered queries, or — for very small filtered subsets — switching to a brute-force scan of just the matching points, which Qdrant can do automatically when the filtered subset is small enough.

**Q: What's the difference between `ef_construction` and `ef`, and which one would you tune first in production?**
A: `ef_construction` is fixed at index build time — it controls how thoroughly neighbors are searched when building the graph, affecting index quality permanently. `ef` is a per-query parameter — it controls how thoroughly the graph is traversed at search time, affecting recall and latency for each query without touching the index. Tune `ef` first in production, because it's a zero-cost change (no rebuild, takes effect immediately) that directly improves recall. Only increase `ef_construction` and rebuild the index if the index quality itself is the bottleneck — which requires a controlled experiment comparing recall at high `ef` against an index rebuilt with higher `ef_construction`.

### Hidden Concepts Worth Knowing

- **Small world graphs**: the "small world" property (from network science) means that in a well-connected graph, any two nodes can be reached from each other in a small number of hops — like the "six degrees of separation" idea. HNSW deliberately constructs a graph with this property so that traversal from any starting point to any target can complete in logarithmic hops rather than linear ones.
- **The curse of dimensionality and why it doesn't kill HNSW**: in very high dimensions, distances between random points converge — most points are "about equally far" from any query. HNSW partially sidesteps this because it builds connections based on relative proximity during construction, not random sampling — the graph structure captures local neighborhood relationships that remain meaningful even in high dimensions.
- **HNSW memory layout**: the graph edges (which node connects to which) are stored separately from the vectors themselves. Qdrant can store vectors on disk while keeping the graph structure in memory, enabling larger-than-RAM corpora at the cost of disk I/O for vector reads during search — controlled by `on_disk=True` in the collection config.

### Revision Summary

> An index exists to make nearest-neighbor search sub-linear — instead of comparing a query vector to every stored vector, an index lets search skip most comparisons by navigating a pre-built structure. HNSW builds a multi-layer graph where top layers enable fast coarse navigation to the right region of the vector space, and the bottom layer enables precise local search once there. The trade-off is exact vs. approximate: HNSW occasionally misses the single true nearest neighbor, which is almost always acceptable for RAG in exchange for orders-of-magnitude faster search. Key tuning knobs: `m` (graph density, set at build time), `ef_construction` (build quality, set at build time), `ef` (search recall, tunable per query without rebuilding).

In [1]:
"""
qdrant_hnsw.py
----------------
Demonstrates HNSW index parameter control in Qdrant.
Uses :memory: mode -- no Docker required.

Install: pip install qdrant-client sentence-transformers
"""

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,       # tells Qdrant how to measure closeness between vectors
    VectorParams,   # defines the vector size and distance metric for a collection
    HnswConfigDiff, # lets us override the default HNSW index parameters
    PointStruct,    # the object that holds one point: id + vector + payload
    Filter,         # wraps one or more conditions for a filtered search
    FieldCondition, # one condition: "this payload field must match this value"
    MatchValue,     # the "must equal this exact value" part of a FieldCondition
    SearchParams,   # lets us control ef (search-time recall vs. speed trade-off)
)
from sentence_transformers import SentenceTransformer

COLLECTION_NAME = "fd_hnsw_demo"
VECTOR_SIZE = 384       # must match the embedding model's output dimension
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

# :memory: means no Docker, no server -- everything lives in this Python process
client = QdrantClient(":memory:")

# load the embedding model once -- reused for every encode() call below
model = SentenceTransformer(MODEL_NAME)

# small set of sample chunks -- in a real pipeline these come from chunker.py
CHUNKS = [
    {"text": "Premature withdrawal incurs a 1 percent penalty on the applicable rate.", "source": "FD_Policy", "doc_type": "policy"},
    {"text": "This penalty does not apply if the FD is closed due to the death of the depositor.", "source": "FD_Policy", "doc_type": "policy"},
    {"text": "Senior citizens receive an additional 0.5 percent interest on all tenures.", "source": "FD_Product_Guide", "doc_type": "product"},
    {"text": "What is the minimum amount to open an FD? The minimum deposit is Rs 25,000.", "source": "FD_FAQ", "doc_type": "faq"},
    {"text": "Can I withdraw my FD before maturity? Yes, subject to a penalty.", "source": "FD_FAQ", "doc_type": "faq"},
    {"text": "The FD interest rate for 24 months is 7.1 percent per annum.", "source": "FD_Product_Guide", "doc_type": "product"},
    {"text": "Nomination facility is available for all Fixed Deposit accounts.", "source": "FD_Policy", "doc_type": "policy"},
    {"text": "TDS is deducted at source if interest income exceeds Rs 40,000 per year.", "source": "FD_Policy", "doc_type": "policy"},
]


def create_hnsw_collection(m: int = 16, ef_construction: int = 100) -> None:
    # check what collections already exist so we don't crash on a duplicate name
    existing = [c.name for c in client.get_collections().collections]

    # delete the old collection if it exists -- clean slate for this demo
    if COLLECTION_NAME in existing:
        client.delete_collection(COLLECTION_NAME)

    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=VECTOR_SIZE,       # every vector in this collection must be 384 floats
            distance=Distance.COSINE,  # cosine similarity -- right choice for normalized vectors
        ),
        hnsw_config=HnswConfigDiff(
            m=m,                    # edges per node -- higher = better recall, more memory
            ef_construct=ef_construction,  # build-time search depth -- higher = better index quality
        ),
    )
    print(f"Created collection with HNSW m={m}, ef_construction={ef_construction}")


def upsert_chunks() -> None:
    # extract just the text strings so we can embed them all in one batch call
    texts = [c["text"] for c in CHUNKS]

    # embed all texts at once -- normalize_embeddings=True makes dot product == cosine similarity
    embeddings = model.encode(texts, normalize_embeddings=True)

    # build a PointStruct for each chunk: id + vector + payload (metadata)
    points = [
        PointStruct(
            id=i,                           # simple integer ID -- fine for a demo
            vector=embeddings[i].tolist(),  # numpy array -> plain Python list for serialization
            payload={
                "text": CHUNKS[i]["text"],      # store original text so search results are self-contained
                "source": CHUNKS[i]["source"],  # which source file this chunk came from
                "doc_type": CHUNKS[i]["doc_type"],  # used for filtering later
            },
        )
        for i in range(len(CHUNKS))
    ]

    # upsert = insert if new, replace if ID already exists
    client.upsert(collection_name=COLLECTION_NAME, points=points)

    # confirm how many points are now in the collection
    info = client.get_collection(COLLECTION_NAME)
    print(f"Upserted {info.points_count} points")


def search_unfiltered(query: str, k: int = 3, ef: int = 128) -> None:
    # embed the query with the same model used at ingest time -- must always match
    query_vector = model.encode(query, normalize_embeddings=True).tolist()

    # query_points replaces the old client.search() which was removed in qdrant-client >= 1.9
    # .points unwraps the result object to get the actual list of scored hits
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,         # the embedded query vector to search against
        limit=k,                    # return the top-k most similar points
        search_params=SearchParams(
            hnsw_ef=ef              # higher ef = more thorough search = better recall, slower
        ),
        with_payload=True,          # include the payload (text, source, doc_type) in results
    ).points                        # .points extracts the list from the response wrapper

    print(f"\nUnfiltered search: '{query}' (ef={ef})")
    for r in results:
        # r.score is the cosine similarity -- closer to 1.0 = more similar
        print(f"  score={r.score:.4f} | {r.payload['text'][:70]}")


def search_filtered(query: str, doc_type: str, k: int = 3) -> None:
    # same embedding step as unfiltered search -- model must always be the same
    query_vector = model.encode(query, normalize_embeddings=True).tolist()

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=Filter(
            must=[                          # ALL conditions in must[] must be true
                FieldCondition(
                    key="doc_type",         # the payload field to filter on
                    match=MatchValue(value=doc_type),  # must equal this exact value
                )
            ]
        ),
        limit=k,
        with_payload=True,
    ).points

    print(f"\nFiltered search (doc_type='{doc_type}'): '{query}'")
    for r in results:
        print(f"  score={r.score:.4f} | [{r.payload['doc_type']}] {r.payload['text'][:60]}")


# ----------------------------------------------------------------------
# Run all demos in order
# ----------------------------------------------------------------------

# step 1: create the collection with explicit HNSW settings
create_hnsw_collection(m=16, ef_construction=100)

# step 2: embed and store all sample chunks
upsert_chunks()

# step 3: search across all doc types -- no filter
search_unfiltered("What happens if I close my FD early?", k=3, ef=64)

# step 4: same query but only look inside FAQ documents
search_filtered("What happens if I close my FD early?", doc_type="faq", k=2)

# step 5: different query, only look inside policy documents
search_filtered("penalty for early closure", doc_type="policy", k=3)

# step 6: compare low ef vs high ef on the same query
# at 8 points the scores will be identical -- the difference shows at scale
print("\n--- ef comparison (difference visible at scale, not at 8 points) ---")
search_unfiltered("senior citizen interest rate", k=2, ef=10)   # fast, lower recall at scale
search_unfiltered("senior citizen interest rate", k=2, ef=200)  # slower, higher recall at scale


Created collection with HNSW m=16, ef_construction=100
Upserted 8 points

Unfiltered search: 'What happens if I close my FD early?' (ef=64)
  score=0.5106 | Can I withdraw my FD before maturity? Yes, subject to a penalty.
  score=0.4080 | This penalty does not apply if the FD is closed due to the death of th
  score=0.3879 | The FD interest rate for 24 months is 7.1 percent per annum.

Filtered search (doc_type='faq'): 'What happens if I close my FD early?'
  score=0.5106 | [faq] Can I withdraw my FD before maturity? Yes, subject to a pena
  score=0.3675 | [faq] What is the minimum amount to open an FD? The minimum deposi

Filtered search (doc_type='policy'): 'penalty for early closure'
  score=0.6175 | [policy] Premature withdrawal incurs a 1 percent penalty on the appli
  score=0.4741 | [policy] This penalty does not apply if the FD is closed due to the d
  score=0.0928 | [policy] TDS is deducted at source if interest income exceeds Rs 40,0

--- ef comparison (difference visible at 